# Notebook 05: Evaluation and Validation

## RustWeatherML - Weather Prediction System in Rust

This notebook covers:
1. Classification metrics (accuracy, precision, recall, F1)
2. Regression metrics (RMSE, MAE, R², MAPE)
3. Confusion matrices
4. Error analysis by city
5. Seasonal performance analysis
6. Final model validation on test set

**Input**: Trained models and test data

**Output**: Comprehensive evaluation report

---
## 1. Setup Dependencies

In [ ]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde"] }
:dep linfa = "0.7"
:dep linfa-trees = "0.7"
:dep smartcore = "0.3"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [ ]:
use polars::prelude::*;
use ndarray::{Array1, Array2};
use linfa::prelude::*;
use linfa_trees::{DecisionTree, SplitQuality};
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::ensemble::random_forest_classifier::RandomForestClassifier;
use std::collections::HashMap;

println!("Dependencies loaded!");

---
## 2. Evaluation Metrics Implementation

In [ ]:
/// Classification Metrics

fn accuracy(y_true: &[u32], y_pred: &[u32]) -> f64 {
    let correct = y_true.iter().zip(y_pred.iter())
        .filter(|(t, p)| t == p).count();
    correct as f64 / y_true.len() as f64
}

fn precision_recall_f1(y_true: &[u32], y_pred: &[u32], positive_class: u32) -> (f64, f64, f64) {
    let mut tp = 0;
    let mut fp = 0;
    let mut fn_ = 0;
    
    for (t, p) in y_true.iter().zip(y_pred.iter()) {
        if *p == positive_class && *t == positive_class {
            tp += 1;
        } else if *p == positive_class && *t != positive_class {
            fp += 1;
        } else if *p != positive_class && *t == positive_class {
            fn_ += 1;
        }
    }
    
    let precision = if tp + fp > 0 { tp as f64 / (tp + fp) as f64 } else { 0.0 };
    let recall = if tp + fn_ > 0 { tp as f64 / (tp + fn_) as f64 } else { 0.0 };
    let f1 = if precision + recall > 0.0 { 
        2.0 * precision * recall / (precision + recall) 
    } else { 
        0.0 
    };
    
    (precision, recall, f1)
}

fn confusion_matrix(y_true: &[u32], y_pred: &[u32], n_classes: usize) -> Vec<Vec<usize>> {
    let mut matrix = vec![vec![0usize; n_classes]; n_classes];
    
    for (t, p) in y_true.iter().zip(y_pred.iter()) {
        matrix[*t as usize][*p as usize] += 1;
    }
    
    matrix
}

println!("Classification metrics defined!");

In [ ]:
/// Regression Metrics

fn rmse(y_true: &[f64], y_pred: &[f64]) -> f64 {
    let mse: f64 = y_true.iter().zip(y_pred.iter())
        .map(|(t, p)| (t - p).powi(2)).sum::<f64>() / y_true.len() as f64;
    mse.sqrt()
}

fn mae(y_true: &[f64], y_pred: &[f64]) -> f64 {
    y_true.iter().zip(y_pred.iter())
        .map(|(t, p)| (t - p).abs()).sum::<f64>() / y_true.len() as f64
}

fn r_squared(y_true: &[f64], y_pred: &[f64]) -> f64 {
    let mean: f64 = y_true.iter().sum::<f64>() / y_true.len() as f64;
    let ss_tot: f64 = y_true.iter().map(|t| (t - mean).powi(2)).sum();
    let ss_res: f64 = y_true.iter().zip(y_pred.iter())
        .map(|(t, p)| (t - p).powi(2)).sum();
    
    if ss_tot == 0.0 { 0.0 } else { 1.0 - (ss_res / ss_tot) }
}

fn mape(y_true: &[f64], y_pred: &[f64]) -> f64 {
    let valid: Vec<(f64, f64)> = y_true.iter().zip(y_pred.iter())
        .filter(|(t, _)| t.abs() > 0.001)  // Avoid division by zero
        .map(|(t, p)| (*t, *p))
        .collect();
    
    if valid.is_empty() { return 0.0; }
    
    valid.iter()
        .map(|(t, p)| ((t - p) / t).abs())
        .sum::<f64>() / valid.len() as f64 * 100.0
}

println!("Regression metrics defined!");

---
## 3. Load Data and Models

In [ ]:
// Load test data (final held-out set)
let test_df = LazyFrame::scan_parquet("../data/features/test.parquet", Default::default())
    .unwrap().collect().unwrap();

let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();

println!("Data loaded:");
println!("  Train: {} rows", train_df.height());
println!("  Test:  {} rows", test_df.height());

In [ ]:
// Reuse helper functions and feature columns from previous notebooks
let feature_cols: Vec<&str> = vec![
    "temperature_2m", "apparent_temperature", "dewpoint_2m",
    "precipitation", "rain", "snowfall",
    "windspeed_10m", "windgusts_10m", "winddirection_10m",
    "pressure_msl", "surface_pressure", "cloudcover", "visibility",
    "shortwave_radiation", "direct_radiation", "relativehumidity_2m",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
    "temp_lag_1h", "temp_lag_6h", "temp_lag_12h", "temp_lag_24h",
    "pressure_lag_1h", "pressure_lag_6h", "pressure_lag_24h",
    "humidity_lag_1h", "humidity_lag_6h",
    "wind_lag_1h", "wind_lag_6h", "precip_lag_1h",
    "temp_change_1h", "temp_change_6h", "temp_change_24h",
    "pressure_change_1h", "pressure_change_6h", "pressure_change_24h",
    "humidity_change_1h",
    "latitude", "longitude",
];

fn df_to_array2(df: &DataFrame, cols: &[&str]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    for col_name in cols {
        let col = df.column(*col_name).unwrap();
        let values = col.cast(&DataType::Float64).unwrap().f64().unwrap().to_vec();
        for val in values { data.push(val.unwrap_or(0.0)); }
    }
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}

fn df_to_array1(df: &DataFrame, col_name: &str) -> Array1<f64> {
    let col = df.column(col_name).unwrap();
    let values: Vec<f64> = col.cast(&DataType::Float64).unwrap()
        .f64().unwrap().to_vec().into_iter().map(|v| v.unwrap_or(0.0)).collect();
    Array1::from_vec(values)
}

fn ndarray_to_dense_matrix(arr: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&arr.outer_iter().map(|row| row.to_vec()).collect::<Vec<_>>())
}

println!("Helper functions defined!");

In [ ]:
// Prepare clean data
let train_clean = train_df.clone().lazy()
    .filter(col("temp_next_24h").is_not_null().and(col("temp_lag_24h").is_not_null()))
    .collect().unwrap();

let test_clean = test_df.clone().lazy()
    .filter(col("temp_next_24h").is_not_null().and(col("temp_lag_24h").is_not_null()))
    .collect().unwrap();

let X_train = df_to_array2(&train_clean, &feature_cols);
let X_test = df_to_array2(&test_clean, &feature_cols);

let y_train_temp = df_to_array1(&train_clean, "temp_next_24h");
let y_test_temp = df_to_array1(&test_clean, "temp_next_24h");

let y_train_rain: Vec<u32> = df_to_array1(&train_clean, "will_rain").iter().map(|&v| v as u32).collect();
let y_test_rain: Vec<u32> = df_to_array1(&test_clean, "will_rain").iter().map(|&v| v as u32).collect();

println!("Data prepared:");
println!("  X_train: {:?}", X_train.shape());
println!("  X_test:  {:?}", X_test.shape());

---
## 4. Train Final Models

In [ ]:
// Load best hyperparameters
let best_params_str = std::fs::read_to_string("../models/best_hyperparameters.json")
    .unwrap_or("{\"hyperparameters\": {\"max_depth\": 10, \"min_weight_split\": 10}}".to_string());
let best_params: serde_json::Value = serde_json::from_str(&best_params_str).unwrap();

let max_depth = best_params["hyperparameters"]["max_depth"].as_u64().unwrap_or(10) as usize;
let min_weight = best_params["hyperparameters"]["min_weight_split"].as_f64().unwrap_or(10.0);

println!("Using best hyperparameters:");
println!("  max_depth: {}", max_depth);
println!("  min_weight_split: {}", min_weight);

In [ ]:
// Train Decision Tree for regression
let train_dataset = linfa::Dataset::new(X_train.clone(), y_train_temp.clone());

let dt_regressor = DecisionTree::params()
    .max_depth(Some(max_depth))
    .min_weight_split(min_weight)
    .split_quality(SplitQuality::Variance)
    .fit(&train_dataset)
    .expect("Failed to fit");

println!("✓ Decision Tree Regressor trained");

In [ ]:
// Train Random Forest for classification
let X_train_sm = ndarray_to_dense_matrix(&X_train);
let X_test_sm = ndarray_to_dense_matrix(&X_test);

let rf_classifier = RandomForestClassifier::fit(
    &X_train_sm,
    &y_train_rain,
    Default::default()
).expect("Failed to fit RF");

println!("✓ Random Forest Classifier trained");

---
## 5. Regression Evaluation

In [ ]:
println!("=== REGRESSION EVALUATION: Temperature 24h Forecast ===");

let y_pred_temp = dt_regressor.predict(&X_test);
let y_pred_temp_vec: Vec<f64> = y_pred_temp.to_vec();
let y_test_temp_vec: Vec<f64> = y_test_temp.to_vec();

let test_rmse = rmse(&y_test_temp_vec, &y_pred_temp_vec);
let test_mae = mae(&y_test_temp_vec, &y_pred_temp_vec);
let test_r2 = r_squared(&y_test_temp_vec, &y_pred_temp_vec);
let test_mape = mape(&y_test_temp_vec, &y_pred_temp_vec);

println!("\nTest Set Metrics:");
println!("  RMSE: {:.4}°C", test_rmse);
println!("  MAE:  {:.4}°C", test_mae);
println!("  R²:   {:.4}", test_r2);
println!("  MAPE: {:.2}%", test_mape);

println!("\nInterpretation:");
if test_rmse < 2.0 {
    println!("  ✓ Excellent: RMSE < 2°C");
} else if test_rmse < 3.0 {
    println!("  ✓ Good: RMSE < 3°C");
} else if test_rmse < 5.0 {
    println!("  ◐ Acceptable: RMSE < 5°C");
} else {
    println!("  ⚠ Needs improvement: RMSE >= 5°C");
}

In [ ]:
// Error distribution analysis
println!("\n--- Error Distribution ---");

let errors: Vec<f64> = y_test_temp_vec.iter().zip(y_pred_temp_vec.iter())
    .map(|(t, p)| t - p).collect();

let mean_error: f64 = errors.iter().sum::<f64>() / errors.len() as f64;
let std_error: f64 = (errors.iter().map(|e| (e - mean_error).powi(2)).sum::<f64>() 
                      / errors.len() as f64).sqrt();

let mut sorted_errors = errors.clone();
sorted_errors.sort_by(|a, b| a.partial_cmp(b).unwrap());

let p5 = sorted_errors[(errors.len() as f64 * 0.05) as usize];
let p25 = sorted_errors[(errors.len() as f64 * 0.25) as usize];
let median = sorted_errors[errors.len() / 2];
let p75 = sorted_errors[(errors.len() as f64 * 0.75) as usize];
let p95 = sorted_errors[(errors.len() as f64 * 0.95) as usize];

println!("  Mean Error:   {:.4}°C (bias)", mean_error);
println!("  Std Error:    {:.4}°C", std_error);
println!("  5th percentile:  {:.2}°C", p5);
println!("  25th percentile: {:.2}°C", p25);
println!("  Median:          {:.2}°C", median);
println!("  75th percentile: {:.2}°C", p75);
println!("  95th percentile: {:.2}°C", p95);

---
## 6. Classification Evaluation

In [ ]:
println!("\n=== CLASSIFICATION EVALUATION: Rain Prediction ===");

let y_pred_rain = rf_classifier.predict(&X_test_sm).expect("Failed to predict");

let acc = accuracy(&y_test_rain, &y_pred_rain);
let (prec, rec, f1) = precision_recall_f1(&y_test_rain, &y_pred_rain, 1);  // 1 = rain

println!("\nTest Set Metrics (Rain = Positive Class):");
println!("  Accuracy:  {:.2}%", acc * 100.0);
println!("  Precision: {:.2}%", prec * 100.0);
println!("  Recall:    {:.2}%", rec * 100.0);
println!("  F1 Score:  {:.4}", f1);

In [ ]:
// Confusion Matrix
println!("\n--- Confusion Matrix ---");

let cm = confusion_matrix(&y_test_rain, &y_pred_rain, 2);

println!("\n                 Predicted");
println!("              No Rain    Rain");
println!("         ┌──────────┬──────────┐");
println!("Actual   │          │          │");
println!("No Rain  │ {:>8} │ {:>8} │  (TN/FP)", cm[0][0], cm[0][1]);
println!("         ├──────────┼──────────┤");
println!("Rain     │ {:>8} │ {:>8} │  (FN/TP)", cm[1][0], cm[1][1]);
println!("         └──────────┴──────────┘");

let tn = cm[0][0];
let fp = cm[0][1];
let fn_ = cm[1][0];
let tp = cm[1][1];

println!("\nMetrics from confusion matrix:");
println!("  True Negatives (TN):  {} (correctly predicted no rain)", tn);
println!("  False Positives (FP): {} (predicted rain when none)", fp);
println!("  False Negatives (FN): {} (missed rain events)", fn_);
println!("  True Positives (TP):  {} (correctly predicted rain)", tp);

---
## 7. Error Analysis by City

In [ ]:
println!("\n=== ERROR ANALYSIS BY CITY ===");

// Get city names from test data
let cities: Vec<String> = test_clean.column("city").unwrap()
    .str().unwrap()
    .into_iter()
    .map(|s| s.unwrap_or("").to_string())
    .collect();

// Group errors by city
let mut city_errors: HashMap<String, Vec<f64>> = HashMap::new();

for (i, city) in cities.iter().enumerate() {
    let error = (y_test_temp_vec[i] - y_pred_temp_vec[i]).abs();
    city_errors.entry(city.clone()).or_insert_with(Vec::new).push(error);
}

// Calculate RMSE per city
let mut city_rmse: Vec<(String, f64, usize)> = city_errors.iter()
    .map(|(city, errors)| {
        let mse = errors.iter().map(|e| e.powi(2)).sum::<f64>() / errors.len() as f64;
        (city.clone(), mse.sqrt(), errors.len())
    })
    .collect();

city_rmse.sort_by(|a, b| a.1.partial_cmp(&b.1).unwrap());

println!("\nTemperature Forecast RMSE by City:");
println!("{:<25} {:>10} {:>10}", "City", "RMSE (°C)", "Samples");
println!("{}", "-".repeat(50));

for (city, rmse_val, n) in &city_rmse {
    let indicator = if *rmse_val < 2.0 { "✓" } else if *rmse_val < 3.0 { "◐" } else { "⚠" };
    println!("{:<25} {:>9.4} {:>9} {}", city, rmse_val, n, indicator);
}

println!("\n✓ = Excellent (<2°C), ◐ = Good (<3°C), ⚠ = Needs work (≥3°C)");

---
## 8. Final Summary Report

In [ ]:
println!("\n" + "=".repeat(70).as_str());
println!("                    FINAL EVALUATION REPORT");
println!("=".repeat(70));

println!("\n1. TEMPERATURE FORECASTING (24h ahead)");
println!("   Model: Decision Tree Regressor (linfa)");
println!("   ┌────────────────────────────────────────┐");
println!("   │ Metric          │ Value               │");
println!("   ├────────────────────────────────────────┤");
println!("   │ RMSE            │ {:<19.4}│", test_rmse);
println!("   │ MAE             │ {:<19.4}│", test_mae);
println!("   │ R²              │ {:<19.4}│", test_r2);
println!("   │ MAPE            │ {:<18.2}%│", test_mape);
println!("   └────────────────────────────────────────┘");

println!("\n2. RAIN PREDICTION (Binary Classification)");
println!("   Model: Random Forest Classifier (smartcore)");
println!("   ┌────────────────────────────────────────┐");
println!("   │ Metric          │ Value               │");
println!("   ├────────────────────────────────────────┤");
println!("   │ Accuracy        │ {:<18.2}%│", acc * 100.0);
println!("   │ Precision       │ {:<18.2}%│", prec * 100.0);
println!("   │ Recall          │ {:<18.2}%│", rec * 100.0);
println!("   │ F1 Score        │ {:<19.4}│", f1);
println!("   └────────────────────────────────────────┘");

println!("\n3. BEST/WORST PERFORMING CITIES");
println!("   Best:  {} (RMSE: {:.4}°C)", city_rmse.first().unwrap().0, city_rmse.first().unwrap().1);
println!("   Worst: {} (RMSE: {:.4}°C)", city_rmse.last().unwrap().0, city_rmse.last().unwrap().1);

println!("\n" + "=".repeat(70).as_str());

In [ ]:
// Save evaluation report
let report = serde_json::json!({
    "regression": {
        "task": "temp_24h_forecast",
        "model": "DecisionTree (linfa)",
        "metrics": {
            "rmse": test_rmse,
            "mae": test_mae,
            "r_squared": test_r2,
            "mape": test_mape
        }
    },
    "classification": {
        "task": "rain_prediction",
        "model": "RandomForest (smartcore)",
        "metrics": {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1
        },
        "confusion_matrix": {
            "tn": tn, "fp": fp, "fn": fn_, "tp": tp
        }
    }
});

std::fs::write("../models/evaluation_report.json",
               serde_json::to_string_pretty(&report).unwrap())
    .expect("Failed to save report");

println!("✓ Evaluation report saved to ../models/evaluation_report.json");

---
## 9. Summary

### What we accomplished:
1. ✅ Implemented all evaluation metrics from scratch
2. ✅ Evaluated regression models (RMSE, MAE, R², MAPE)
3. ✅ Evaluated classification models (Accuracy, Precision, Recall, F1)
4. ✅ Created confusion matrix
5. ✅ Analyzed errors by city
6. ✅ Generated comprehensive evaluation report

### Next Steps (Notebook 06):
- Drift detection implementation
- Model monitoring setup
- Production deployment preparation

In [ ]:
println!("\n" + "=".repeat(60).as_str());
println!("Notebook 05 Complete!");
println!("=".repeat(60));
println!("\nProceed to Notebook 06: Drift Detection & Monitoring");